# Unequal Housing Conditions: EDA for Reactive Datasets

## Summary

In April 2022, Health Affairs journal [published research](https://drive.google.com/file/d/1UNsZFoiROnXif1XU6L2_kiELQ-xJmMia/view?usp=share_link) correlating racial makeup of neighborhoods to unequal housing conditions and code enforcement related to asthma triggers. The researchers found that, when analyzing publicly-available 311 data between 2011 and 2021, predominantly white neighborhoods report fewer conditions known to cause or exacerbate asthma, and when they do, they experience faster case closure rates, fewer overdue requests, and higher chances of the request resulting in a repair. 

While the above study was predicated on the publicly-available 311 data where renters call to put in their complaints before the City react (we term this data as reactive dataset), there has been a policy which allows the [City to randomly inspect housing](https://www.boston.gov/departments/inspectional-services/inspecting-housing-boston) to ensure safety and protect the health of the public.

Thus, the inspectional service department developed [an algorithm](https://drive.google.com/file/d/1Cn444cCKTDTU2sqXXCiqzDLha3wGvczA/view?usp=drive_link) to randomly assign houses for inspection. This proactive approach would be analyze to explore and determine the efficacy of the policy as it pertains to ensuring equitable distribution of the City's housing safety protocols. 


## EDA Problem Statement

Datasets: 
* 311 Requests, 2020 - Present
* Reactive Inspections data, 2023 - Present

Explore the different features in the dataset to see if it will good for feature selection and determining feature importance.
* If possible, a predictive model could be developed to using property features, location & other factors as IVs.
* Graph Analysis; to connect different locations using various nodes and edges to gain insights into likely violators.
* Similarity Analysis/CLustering based on different features.



In [1]:
#####  Import libraries for Boston City DoIT Analytics database..
import os 
import civis

# import libraries for data pre-processing
import pandas as pd
import numpy as np
from datetime import datetime
import datetime as datetime

# import libraries for visualization
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from chart_studio import plotly
import plotly.offline as pyoff
import plotly.graph_objs as go
import plotly.express as px


from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=True)        # Display plot on notebook while the kernel is running


# utils
import warnings 
warnings.filterwarnings('ignore')


In [2]:
os.environ['PYTHONPATH']='.'
os.environ['CIVIS_API_KEY']='iNtRtV8mWJYVuDKqr9XULieOXud8XndxiyWpFnSPUxs'

# Define the SQL query for reactive inspections/cases from 2023 - present 
sql_reactive_cases = """
SELECT * FROM isd_internal_data.reactive_cases
WHERE adddttm >= '2023-01-01 00:00:00+00'
"""
# Define the SQL query for reactive violations from 2023 - present 
sql_reactive_violations = """
SELECT * FROM isd_internal_data.reactive_violations 
WHERE adddttm >= '2023-01-01 00:00:00+00'
"""

df_reactive_inspections = civis.io.read_civis_sql(sql_reactive_cases,
                                            database='Boston',
                                            use_pandas=True)
df_reactive_violations = civis.io.read_civis_sql(sql_reactive_violations,
                                            database='Boston',
                                            use_pandas=True)


In [3]:
df_reactive_inspections.head()


,addby,adddttm,apby,apcasekey,apcaseprocessstatekey,apno,casegrp,caseinfostatus,casename,comments,...,empid,emplast,empmid,supr,suprflag,adddttm_fy,full_address,full_address_w_ward,typeno_translation_et,gps_geom_2249
0,135884,2023-07-06 19:42:26.03+00,135884,684003,1120,V684003,BLDG,Open,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2023,726Columbia RD,726Columbia RD 07,NaN,0101000020C908000083F04F35EEA62741687BD3F44572...
1,127656,2023-02-15 15:27:04.923+00,127656,661875,1111,V661875,BLDG,Closed,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2022,5Monmouth ST,5Monmouth ST 01,NaN,0101000020C9080000828405169ED327416163EA964A9C...
2,127656,2023-06-26 15:18:41.637+00,127656,682211,1120,V682211,BLDG,Open,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2022,252Newton ST,252Newton ST 04,NaN,0101000020C908000084D47B5CC57A27419768982A8783...
3,127656,2023-08-22 14:17:32.82+00,127656,694356,1120,V694356,BLDG,Open,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2023,24Cranston ST,24Cranston ST 19,NaN,0101000020C9080000FC5B254B4C412741E5DFBE8E7F72...
4,127656,2023-06-20 13:57:58.097+00,127656,681176,1111,V681176,BLDG,Closed,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2022,73Revere ST,73Revere ST 05,NaN,0101000020C90800007E555487499427417CA7D51A288E...


In [4]:
df_reactive_inspections.tail()


,addby,adddttm,apby,apcasekey,apcaseprocessstatekey,apno,casegrp,caseinfostatus,casename,comments,...,empid,emplast,empmid,supr,suprflag,adddttm_fy,full_address,full_address_w_ward,typeno_translation_et,gps_geom_2249
42881,140965,2023-02-13 17:21:43.007+00,140965,661445,1144,CE661445,,Open,NaN,NaN,...,140965,NaN,NaN,,N,2022,68Calumet ST,68Calumet ST 10,12:21:43,0101000020C90800000340F498574A274197F1E3F7667A...
42882,127656,2023-07-12 16:39:44.74+00,127656,685044,1110,V685044,BLDG,Open,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2023,599 to 601Washington ST,599 to 601Washington ST 17,NaN,0101000020C908000081275DA0558F27411E3F2B3CFF5C...
42883,115680,2023-05-30 11:55:32.133+00,115680,677588,1144,CE677588,,Open,NaN,NaN,...,115680,NaN,NaN,,N,2022,35 to 37Colborne RD,35 to 37Colborne RD 21,07:55:32,0101000020C9080000825EFC74D7F1264102E62DAE9A82...
42884,52099,2023-05-30 11:57:20+00,52099,677589,1144,CE677589,,Open,NaN,NaN,...,052099,Donovan,NaN,,N,2022,3Worcester SQ,3Worcester SQ 08,07:57:20,0101000020C90800007E30442C4D872741B770C9A4EB7D...
42885,115680,2023-06-26 17:04:27.537+00,115680,682252,1144,CE682252,,Open,NaN,NaN,...,115680,NaN,NaN,,N,2022,1Annapolis ST,1Annapolis ST 13,13:04:28,0101000020C9080000FD743E1DAEA32741E75FC3610C70...


In [5]:
df_reactive_violations.tail()


,addby,adddttm,apby,apcasekey,apcaseprocessstatekey,apno,casegrp,caseinfostatus,casename,comments,...,postdir,state,stname,stno,suffix,zip,adddttm_fy,full_address,full_address_w_ward,gps_geom_2249
51301,50764,2023-01-30 12:57:17.92+00,50764,658902,1144,CE658902,,Open,NaN,NaN,...,17,MA,Crowell,9,ST,2124,2022,9Crowell ST,9Crowell ST 17,0101000020C9080000838B45CC587927419B9B63EFC757...
51302,100277,2023-01-30 12:57:21.43+00,100277,658903,1144,CE658903,,Open,NaN,NaN,...,13,MA,Bakersfield,68,ST,2125,2022,68Bakersfield ST,68Bakersfield ST 13,0101000020C908000082686ED87FA52741F8A50243EB6E...
51303,100277,2023-01-30 12:57:21.43+00,100277,658903,1144,CE658903,,Open,NaN,NaN,...,13,MA,Bakersfield,68,ST,2125,2022,68Bakersfield ST,68Bakersfield ST 13,0101000020C908000082686ED87FA52741F8A50243EB6E...
51304,104619,2023-01-30 13:03:44.04+00,104619,658904,1144,CE658904,,Open,NaN,NaN,...,21,MA,Commonwealth,1629,AV,2135,2022,1629Commonwealth AV,1629Commonwealth AV 21,0101000020C90800001AF61B7236F626417A4757E93D82...
51305,104619,2023-01-30 13:03:44.04+00,104619,658904,1144,CE658904,,Open,NaN,NaN,...,21,MA,Commonwealth,1629,AV,2135,2022,1629Commonwealth AV,1629Commonwealth AV 21,0101000020C90800001AF61B7236F626417A4757E93D82...


In [6]:
sub = df_reactive_inspections[['apno','adddttm','scheddttm','compdttm','resultdttm','resdttm','result',
                               'resultby','code','code_caseinspresult','gpsx','gpsy','gps_geom_2249']]


In [7]:
sub.head()

,apno,adddttm,scheddttm,compdttm,resultdttm,resdttm,result,resultby,code,code_caseinspresult,gpsx,gpsy,gps_geom_2249
0,V684003,2023-07-06 19:42:26.03+00,NaN,NaN,NaN,NaN,NaN,NaN,Printing and Service,NaN,775031.104126,2.942092e+06,0101000020C908000083F04F35EEA62741687BD3F44572...
1,V661875,2023-02-15 15:27:04.923+00,NaN,NaN,NaN,2023-02-24 15:50:19+00,NaN,NaN,Closed,NaN,780751.043011,2.963605e+06,0101000020C9080000828405169ED327416163EA964A9C...
2,V682211,2023-06-26 15:18:41.637+00,NaN,NaN,NaN,NaN,NaN,NaN,Printing and Service,NaN,769378.680632,2.950926e+06,0101000020C908000084D47B5CC57A27419768982A8783...
3,V694356,2023-08-22 14:17:32.82+00,NaN,NaN,NaN,NaN,NaN,NaN,Printing and Service,NaN,762022.146769,2.942207e+06,0101000020C9080000FC5B254B4C412741E5DFBE8E7F72...
4,V681176,2023-06-20 13:57:58.097+00,NaN,NaN,NaN,2023-06-30 18:24:45+00,NaN,NaN,Closed,NaN,772644.764315,2.956368e+06,0101000020C90800007E555487499427417CA7D51A288E...


In [8]:
sub.tail()

,apno,adddttm,scheddttm,compdttm,resultdttm,resdttm,result,resultby,code,code_caseinspresult,gpsx,gpsy,gps_geom_2249
42881,CE661445,2023-02-13 17:21:43.007+00,2023-02-13 17:21:43+00,2023-02-13 17:20:00+00,2023-02-13 17:20:00+00,NaN,Violation,140965,Final Reminder,Violation,763179.798738,2.946254e+06,0101000020C90800000340F498574A274197F1E3F7667A...
42882,V685044,2023-07-12 16:39:44.74+00,NaN,NaN,NaN,NaN,NaN,NaN,Intake,NaN,772010.813211,2.931198e+06,0101000020C908000081275DA0558F27411E3F2B3CFF5C...
42883,CE677588,2023-05-30 11:55:32.133+00,2023-05-30 11:55:32+00,2023-05-30 11:54:00+00,2023-05-30 11:54:00+00,NaN,Violation,115680,Final Reminder,Violation,751851.728488,2.950453e+06,0101000020C9080000825EFC74D7F1264102E62DAE9A82...
42884,CE677589,2023-05-30 11:57:20+00,2023-05-30 11:57:20+00,2023-05-30 11:55:00+00,2023-05-30 11:55:00+00,NaN,Violation,052099,Final Reminder,Violation,770982.586458,2.948055e+06,0101000020C90800007E30442C4D872741B770C9A4EB7D...
42885,CE682252,2023-06-26 17:04:27.537+00,2023-06-26 17:04:28+00,2023-06-26 17:03:00+00,2023-06-26 17:03:00+00,NaN,Violation,115680,Final Reminder,Violation,774615.057117,2.940953e+06,0101000020C9080000FD743E1DAEA32741E75FC3610C70...


### EDA

In [9]:
sub.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42886 entries, 0 to 42885
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   apno                 42886 non-null  object 
 1   adddttm              42886 non-null  object 
 2   scheddttm            40423 non-null  object 
 3   compdttm             39902 non-null  object 
 4   resultdttm           40429 non-null  object 
 5   resdttm              4537 non-null   object 
 6   result               42728 non-null  object 
 7   resultby             42728 non-null  object 
 8   code                 42886 non-null  object 
 9   code_caseinspresult  42728 non-null  object 
 10  gpsx                 42861 non-null  float64
 11  gpsy                 42861 non-null  float64
 12  gps_geom_2249        42861 non-null  object 
dtypes: float64(2), object(11)
memory usage: 4.3+ MB


* The missing records in the 'compdttm' means an inspection has not been completed in those units, explore its details..

In [10]:
# Count the number of empty rows in the 'compdttm' column.
empty_rows = sub['compdttm'].isnull().sum()

# Count the number of rows with empty strings in the 'compdttm' column.
empty_strings = (sub['compdttm'] == '').sum()

# Calculate the total number of empty rows or strings in the 'compdttm' column.
total_empty = empty_rows + empty_strings
total_empty


2984

#### Uninspected Units
**2657** reported cases were not inspected

In [11]:
# Drop rows where 'compdttm' column is either empty or contains an empty string.
sub.dropna(subset=['compdttm'], inplace=True)
sub.tail()


,apno,adddttm,scheddttm,compdttm,resultdttm,resdttm,result,resultby,code,code_caseinspresult,gpsx,gpsy,gps_geom_2249
42880,CE661444,2023-02-13 17:18:24.547+00,2023-02-13 17:18:25+00,2023-02-13 17:17:00+00,2023-02-13 17:17:00+00,NaN,Violation,140965,Final Reminder,Violation,763932.992147,2.945645e+06,0101000020C908000005B7FAFB3950274120CD9E9A3679...
42881,CE661445,2023-02-13 17:21:43.007+00,2023-02-13 17:21:43+00,2023-02-13 17:20:00+00,2023-02-13 17:20:00+00,NaN,Violation,140965,Final Reminder,Violation,763179.798738,2.946254e+06,0101000020C90800000340F498574A274197F1E3F7667A...
42883,CE677588,2023-05-30 11:55:32.133+00,2023-05-30 11:55:32+00,2023-05-30 11:54:00+00,2023-05-30 11:54:00+00,NaN,Violation,115680,Final Reminder,Violation,751851.728488,2.950453e+06,0101000020C9080000825EFC74D7F1264102E62DAE9A82...
42884,CE677589,2023-05-30 11:57:20+00,2023-05-30 11:57:20+00,2023-05-30 11:55:00+00,2023-05-30 11:55:00+00,NaN,Violation,052099,Final Reminder,Violation,770982.586458,2.948055e+06,0101000020C90800007E30442C4D872741B770C9A4EB7D...
42885,CE682252,2023-06-26 17:04:27.537+00,2023-06-26 17:04:28+00,2023-06-26 17:03:00+00,2023-06-26 17:03:00+00,NaN,Violation,115680,Final Reminder,Violation,774615.057117,2.940953e+06,0101000020C9080000FD743E1DAEA32741E75FC3610C70...


In [12]:
# Look at the unique values from the 'result' column
unique_results = sub['result'].unique()
unique_results


array(['Violation', 'OFA', 'Violations', 'NoViol', 'Served', 'ServedM',
       'OFANFC', 'NotServed', 'RP', 'Withdrawn', 'CourtReins',
       'Corrected', 'Condemn', 'UnitVacant', 'NollePros', 'Send Legal',
       'Hearing', 'Failed', ' '], dtype=object)

In [13]:
# Look at the unique values from the 'code_caseinspresult' column
unique_code = sub['code_caseinspresult'].unique()
unique_code


array(['Violation', 'OFA', 'Violations', 'NoViol', 'Served', 'ServedM',
       'OFANFC', 'NotServed', 'RP', 'Withdrawn', 'CourtReins',
       'Corrected', 'Condemn', 'UnitVacant', 'NollePros', 'Send Legal',
       'Hearing', 'Failed', ' '], dtype=object)

### How to Categorized Code Result?
* Should we have three categories:
    1. Fail: Violation(s), Failed, Condemn, RP, CourtReins, 
    2. Pass: NoViol, Withdrawn,  Corrected, 
    3. IC: 'OFA', 'OFANFC', UnitVacant,**
* What about (CourtReins, NollePros, RP, OFA, etc.
* Drop:  Served, ServedM

In [14]:
# Create new categories for Pass, Fail and IC.
sub['result'] = sub['result'].replace({'NoViol': 'Pass', 'Withdrawn': 'Pass', 'Corrected': 'Pass'})
sub['result'] = sub['result'].replace({'Failed': 'Fail', 'Violation': 'Fail', 'Violations': 'Fail',
                                       'Condemn': 'Fail', 'RP': 'Fail', 'CourtReins': 'Fail'})
sub['result'] = sub['result'].replace({'OFA': 'IC', 'OFANFC': 'IC', 'UnitVacant': 'IC'})


In [15]:
# List of valid result categories
valid_results = ['Pass', 'Fail', 'IC']

# Filter the DataFrame to keep only valid result categories
sub = sub[sub['result'].isin(valid_results)]
unique_results = sub['result'].unique()
unique_results


array(['Fail', 'IC', 'Pass'], dtype=object)

In [16]:
# Convert 'Fail' to 0 and 'Pass' to 1 and 'IC' to 2
sub['result'] = sub['result'].map({'Fail': 0, 'Pass': 1, 'IC': 2})

#Print the unique values in the 'result' column to verify the changes
unique_results = sub['result'].unique()
unique_results


array([0, 2, 1], dtype=int64)

In [17]:
# Calculate the count and percentage of the results

result_counts = sub['result'].value_counts()
result_percentages = sub['result'].value_counts(normalize=True) * 100

# Create a DataFrame to display the result
result_table = pd.DataFrame({'Count': result_counts, 'Percentage': result_percentages})

# Format the percentage column to display with two decimal places
result_table['Percentage'] = result_table['Percentage'].map('{:.2f}%'.format)

# Display the result table
print(result_table)


        Count Percentage
result                  
0       36343     94.22%
2        1349      3.50%
1         881      2.28%


For reactive case, 94% failed inspections, which means renters are mostly right, with only 2.27% passing inspections and 3.55& on incomplete.

#### Time Duration Analysis
* For case duration: Using 'adddttm' for the date the inspection was determined and 'resultdttm' for when the inspection was resulted.
**resdttm**

In [18]:
# Drop rows with missing values in the 'scheddttm' column
sub = sub.dropna(subset=['scheddttm'])
sub = sub.dropna(subset=['compdttm'])
sub = sub.dropna(subset=['scheddttm'])
sub = sub.dropna(subset=['resdttm'])


In [19]:
from dateutil import parser

# Convert 'adddtttm', 'scheddttm', 'compdttm' and 'resultdttm' columns to datetime format without seconds and milliseconds.
sub['adddttm'] = sub['adddttm'].apply(lambda x: parser.parse(x).strftime('%Y-%m-%d %H:%M'))
sub['scheddttm'] = sub['scheddttm'].apply(lambda x: parser.parse(x).strftime('%Y-%m-%d %H:%M'))
sub['compdttm'] = sub['compdttm'].apply(lambda x: parser.parse(x).strftime('%Y-%m-%d %H:%M'))
sub['resdttm'] = sub['resdttm'].apply(lambda x: parser.parse(x).strftime('%Y-%m-%d %H:%M'))

# Calculate the time difference between when the case was reported 'adddttm', scheduled for inspection, completed and resulted.
sub['case_to_sched_time'] = pd.to_datetime(sub['scheddttm'], format='%Y-%m-%d %H:%M') - pd.to_datetime(sub['adddttm'], format='%Y-%m-%d %H:%M')
sub['case_to_comp_time'] = pd.to_datetime(sub['compdttm'], format='%Y-%m-%d %H:%M') - pd.to_datetime(sub['adddttm'], format='%Y-%m-%d %H:%M')
sub['case_to_result_time'] = pd.to_datetime(sub['resdttm'], format='%Y-%m-%d %H:%M') - pd.to_datetime(sub['adddttm'], format='%Y-%m-%d %H:%M')
sub['sched_to_comp_time'] = pd.to_datetime(sub['compdttm'], format='%Y-%m-%d %H:%M') - pd.to_datetime(sub['scheddttm'], format='%Y-%m-%d %H:%M')
sub['sched_to_result_time'] = pd.to_datetime(sub['resdttm'], format='%Y-%m-%d %H:%M') - pd.to_datetime(sub['scheddttm'], format='%Y-%m-%d %H:%M')

# Convert the time difference to days and hours for case reported and schedule date time
sub['case_to_sched_days'] = sub['case_to_sched_time'] / pd.Timedelta(days=1)
sub['case_to_sched_hours'] = sub['case_to_sched_time'] / pd.Timedelta(hours=1)

# Convert the time difference to days and hours for case reported and inspection completed date time
sub['case_to_comp_days'] = sub['case_to_comp_time'] / pd.Timedelta(days=1)
sub['case_to_comp_hours'] = sub['case_to_comp_time'] / pd.Timedelta(hours=1)

# Convert the time difference to days and hours for case reported and inspection resulted date time
sub['case_to_result_days'] = sub['case_to_result_time'] / pd.Timedelta(days=1)
sub['case_to_result_hours'] = sub['case_to_result_time'] / pd.Timedelta(hours=1)

# Convert the time difference to days and hours for case inspection scheduled and completed date time
sub['sched_to_comp_days'] = sub['sched_to_comp_time'] / pd.Timedelta(days=1)
sub['sched_to_comp_hours'] = sub['sched_to_comp_time'] / pd.Timedelta(hours=1)

# Convert the time difference to days and hours for case inspection scheduled and resulted date time
sub['sched_to_result_time'] = sub['sched_to_result_time'] / pd.Timedelta(days=1)
# sub['sched_to_result_time'] = sub['sched_to_result_time'] / pd.Timedelta(hours=1)


In [20]:
sub.head()


,apno,adddttm,scheddttm,compdttm,resultdttm,resdttm,result,resultby,code,code_caseinspresult,...,sched_to_comp_time,sched_to_result_time,case_to_sched_days,case_to_sched_hours,case_to_comp_days,case_to_comp_hours,case_to_result_days,case_to_result_hours,sched_to_comp_days,sched_to_comp_hours
109,HVIOL-654697,2023-01-03 17:37,2023-04-05 14:00,2023-04-05 14:39,2023-04-06 11:58:00+00,2023-04-21 12:42,2,083267,Closed,OFA,...,0 days 00:39:00,15.945833,91.849306,2204.383333,91.876389,2205.033333,107.795139,2587.083333,0.027083,0.650000
129,HVIOL-656778,2023-01-17 16:01,2023-01-26 18:00,2023-01-26 20:00,2023-01-27 14:27:00+00,2023-04-18 12:16,0,075702,Closed,Violations,...,0 days 02:00:00,81.761111,9.082639,217.983333,9.165972,219.983333,90.843750,2180.250000,0.083333,2.000000
133,HVIOL-656780,2023-01-17 16:07,2023-01-24 15:30,2023-01-24 15:55,2023-01-25 13:18:00+00,2023-01-25 13:18,1,139460,Closed,NoViol,...,0 days 00:25:00,0.908333,6.974306,167.383333,6.991667,167.800000,7.882639,189.183333,0.017361,0.416667
163,HVIOL-656791,2023-01-17 16:38,2023-01-25 18:00,2023-01-25 19:35,2023-01-25 23:54:00+00,2023-03-06 21:26,0,075702,Closed,Violations,...,0 days 01:35:00,40.143056,8.056944,193.366667,8.122917,194.950000,48.200000,1156.800000,0.065972,1.583333
212,HVIOL-660473,2023-02-07 17:08,2023-02-14 15:30,2023-02-14 16:40,2023-02-14 19:40:00+00,2023-02-14 19:40,1,139460,Closed,NoViol,...,0 days 01:10:00,0.173611,6.931944,166.366667,6.980556,167.533333,7.105556,170.533333,0.048611,1.166667


### Encoding categorical features
#### Reasons for Encoding features or variables
1. Algorithms typically require numeric data. 
2. When a categorical feature is represented with numbers, for example, 1 for female and 2 for male, we need to encode the values so that they are recognized as categorical. 
3. The feature might actually be ordinal, with a discrete number of values that represent some meaningful ranking. Our models need to capture that ranking.
4. A categorical feature might have a large number of values (known as high cardinality), and we might want our encoding to collapse categories.

#### One-hot encoding

One-hot encoding a feature creates a binary vector for each value of that feature. 

So, if a feature, called result, has three unique values, 0, 1, and 2, one-hot encoding creates three binary vectors to represent those values. Recall from above that result 0 is a failed inspection, 1 is passed and 2 is incomplete.

* The first binary vector, which we can call **result_0**, has 1 whenever result has a value of 0, and 0 when it is 1 or 2.
* result_1 and result_2 would be coded similarly. 
* The transformed features, result_0, result_1, and result_2, are often referred to as dummy variables. 


In [21]:
# we import the necessary packages for it
import feature_engine.selection as fesel
from sklearn.model_selection import train_test_split
from feature_engine.encoding import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split


In [22]:
# We reconvert original mapping dictionary of the result column before applying One-Hot Encoding
original_mapping = {0: 'Fail', 1: 'Pass', 2: 'IC'}

# Reverse the mapping and convert back to original strings
sub['result'] = sub['result'].replace(original_mapping)

# Display the updated 'result' column
print(sub['result'])


109        IC
129      Fail
133      Pass
163      Fail
212      Pass
         ... 
42859    Pass
42860      IC
42863    Fail
42867      IC
42876    Fail
Name: result, Length: 2855, dtype: object


In [23]:
feature_col = ['result']

# Create one-hot encoded column for the specified feature column.
one_hot_encoded = pd.get_dummies(sub[feature_col])

# Convert boolean values to integers (0 and 1).
one_hot_encoded = one_hot_encoded.astype(int)

# Concatenate the one-hot encoded columns back to the original 'sub' dataframe.
sub = pd.concat([sub, one_hot_encoded], axis=1)

# Drop the original categorical columns from 'sub' if needed, i will keep it for now
# sub.drop(feature_cols, axis=1, inplace=True)

sub.head()


,apno,adddttm,scheddttm,compdttm,resultdttm,resdttm,result,resultby,code,code_caseinspresult,...,case_to_sched_hours,case_to_comp_days,case_to_comp_hours,case_to_result_days,case_to_result_hours,sched_to_comp_days,sched_to_comp_hours,result_Fail,result_IC,result_Pass
109,HVIOL-654697,2023-01-03 17:37,2023-04-05 14:00,2023-04-05 14:39,2023-04-06 11:58:00+00,2023-04-21 12:42,IC,083267,Closed,OFA,...,2204.383333,91.876389,2205.033333,107.795139,2587.083333,0.027083,0.650000,0,1,0
129,HVIOL-656778,2023-01-17 16:01,2023-01-26 18:00,2023-01-26 20:00,2023-01-27 14:27:00+00,2023-04-18 12:16,Fail,075702,Closed,Violations,...,217.983333,9.165972,219.983333,90.843750,2180.250000,0.083333,2.000000,1,0,0
133,HVIOL-656780,2023-01-17 16:07,2023-01-24 15:30,2023-01-24 15:55,2023-01-25 13:18:00+00,2023-01-25 13:18,Pass,139460,Closed,NoViol,...,167.383333,6.991667,167.800000,7.882639,189.183333,0.017361,0.416667,0,0,1
163,HVIOL-656791,2023-01-17 16:38,2023-01-25 18:00,2023-01-25 19:35,2023-01-25 23:54:00+00,2023-03-06 21:26,Fail,075702,Closed,Violations,...,193.366667,8.122917,194.950000,48.200000,1156.800000,0.065972,1.583333,1,0,0
212,HVIOL-660473,2023-02-07 17:08,2023-02-14 15:30,2023-02-14 16:40,2023-02-14 19:40:00+00,2023-02-14 19:40,Pass,139460,Closed,NoViol,...,166.366667,6.980556,167.533333,7.105556,170.533333,0.048611,1.166667,0,0,1


In [24]:
# Drop the original categorical columns from 'sub' if needed, i will keep it for now
sub.drop(feature_col, axis=1, inplace=True)

sub.tail(15)                                         # look at the last 15 observations


,apno,adddttm,scheddttm,compdttm,resultdttm,resdttm,resultby,code,code_caseinspresult,gpsx,...,case_to_sched_hours,case_to_comp_days,case_to_comp_hours,case_to_result_days,case_to_result_hours,sched_to_comp_days,sched_to_comp_hours,result_Fail,result_IC,result_Pass
42775,HVIOL-676630,2023-05-23 13:16,2023-07-18 14:00,2023-07-18 14:30,2023-07-19 12:33:00+00,2023-07-19 12:34,158090,Closed,NoViol,767730.600311,...,1344.733333,56.051389,1345.233333,56.970833,1367.300000,0.020833,0.500000,0,0,1
42800,HVIOL-677856,2023-05-31 13:21,2023-06-08 15:30,2023-06-08 17:10,2023-06-08 19:38:00+00,2023-06-14 13:31,125041,Closed,Violations,770799.968890,...,194.150000,8.159028,195.816667,14.006944,336.166667,0.069444,1.666667,1,0,0
42801,HVIOL-677857,2023-05-31 13:21,2023-06-07 15:10,2023-06-07 15:55,2023-06-08 11:51:00+00,2023-06-08 11:51,050088,Closed,OFANFC,767124.505590,...,169.816667,7.106944,170.566667,7.937500,190.500000,0.031250,0.750000,0,1,0
42805,HVIOL-677863,2023-05-31 13:34,2023-06-08 15:00,2023-06-08 16:15,2023-06-08 19:39:00+00,2023-07-21 17:52,125041,Closed,Violations,764756.443540,...,193.433333,8.111806,194.683333,51.179167,1228.300000,0.052083,1.250000,1,0,0
42836,HVIOL-670318,2023-04-07 18:46,2023-05-15 12:18,2023-05-15 14:55,2023-05-15 18:31:00+00,2023-04-07 19:19,139460,Closed,Violations,775439.543227,...,905.533333,37.839583,908.150000,0.022917,0.550000,0.109028,2.616667,1,0,0
42840,HVIOL-669741,2023-04-04 17:43,2023-04-05 18:00,2023-04-05 18:45,2023-04-05 23:31:00+00,2023-04-18 12:50,075702,Closed,Violations,768481.684759,...,24.283333,1.043056,25.033333,13.796528,331.116667,0.031250,0.750000,1,0,0
42843,HVIOL-669744,2023-04-04 18:11,2023-04-11 14:00,2023-04-11 15:00,2023-04-12 12:05:00+00,2023-06-14 19:35,083267,Closed,Violations,766226.954421,...,163.816667,6.867361,164.816667,71.058333,1705.400000,0.041667,1.000000,1,0,0
42852,HVIOL-670478,2023-04-10 12:31,2023-04-13 14:00,2023-04-13 16:15,2023-04-13 19:29:00+00,2023-05-22 19:08,139460,Closed,Violations,780070.558050,...,73.483333,3.155556,75.733333,42.275694,1014.616667,0.093750,2.250000,1,0,0
42856,HVIOL-661765,2023-02-14 20:18,2023-04-10 17:00,2023-04-10 17:45,2023-04-10 17:45:00+00,2023-04-11 12:44,148496,Closed,NoViol,763993.551667,...,1316.700000,54.893750,1317.450000,55.684722,1336.433333,0.031250,0.750000,0,0,1
42858,HVIOL-664377,2023-03-02 14:23,2023-04-18 15:00,2023-04-18 16:15,2023-04-18 19:22:00+00,2023-04-18 19:22,130902,Closed,NoViol,767507.951639,...,1128.616667,47.077778,1129.866667,47.207639,1132.983333,0.052083,1.250000,0,0,1


In [25]:
# Export the 'sub' DataFrame to a CSV file
sub.to_csv('reactive_data.csv', index=False)
